In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# CONFIG

INPUT_FILE = "Thailand_Temperature_Clean.csv"

# จำนวนปีที่จะ Forecast
FORECAST_YEARS = 5

# ใช้ข้อมูล 20% ท้าย Dataset เป็น Test
TEST_RATIO = 0.20

RANDOM_STATE = 42

In [3]:
# 1. LOAD DATA

print("=" * 70)
print("THAILAND TEMPERATURE FORECAST")
print("=" * 70)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"ไม่พบไฟล์ {INPUT_FILE}\n"
        "กรุณาวาง CSV ไว้ในโฟลเดอร์เดียวกับ Python Script"
    )

df = pd.read_csv(INPUT_FILE)

print("\nDataset:")
print(f"Rows    : {len(df):,}")
print(f"Columns : {list(df.columns)}")


THAILAND TEMPERATURE FORECAST

Dataset:
Rows    : 63
Columns : ['Year', 'Value']


In [6]:
# 2. FIND YEAR COLUMN

year_candidates = [
    "Year",
    "year",
    "YEAR",
    "Date",
    "date"
]

year_col = None

for col in year_candidates:
    if col in df.columns:
        year_col = col
        break

if year_col is None:

    for col in df.columns:

        if "year" in str(col).lower():
            year_col = col
            break

if year_col is None:
    raise ValueError(
        "ไม่พบ Column ปี เช่น Year หรือ year"
    )

In [8]:
# 3. FIND TEMPERATURE COLUMN

temp_candidates = [
    "Temperature",
    "temperature",
    "Temp",
    "temp",
    "Temperature_C",
    "temperature_c",
    "AverageTemperature",
    "Average_Temperature"
]

temp_col = None

for col in temp_candidates:
    if col in df.columns:
        temp_col = col
        break

if temp_col is None:

    for col in df.columns:

        col_lower = str(col).lower()

        if (
            "temperature" in col_lower
            or "temp" in col_lower
        ):
            temp_col = col
            break

if temp_col is None:
    raise ValueError(
        "ไม่พบ Column อุณหภูมิ เช่น Temperature หรือ Temperature_C"
    )


print("\nDetected columns:")
print(f"Year        : {year_col}")
print(f"Temperature : {temp_col}")


ValueError: ไม่พบ Column อุณหภูมิ เช่น Temperature หรือ Temperature_C

In [ ]:
# 4. PREPARE DATA

data = df[
    [year_col, temp_col]
].copy()

data[year_col] = pd.to_numeric(
    data[year_col],
    errors="coerce"
)

data[temp_col] = pd.to_numeric(
    data[temp_col],
    errors="coerce"
)

# ลบข้อมูลที่ไม่สมบูรณ์
data = data.dropna()

# เปลี่ยนชื่อ
data = data.rename(
    columns={
        year_col: "Year",
        temp_col: "Temperature"
    }
)

# ถ้ามีหลายแถวต่อปี ให้เฉลี่ยเป็นรายปี
data = (
    data
    .groupby("Year", as_index=False)
    ["Temperature"]
    .mean()
)

# เรียงตามปี
data = data.sort_values(
    "Year"
).reset_index(drop=True)


print("\nCleaned data:")
print(
    f"Year range: "
    f"{int(data['Year'].min())} - "
    f"{int(data['Year'].max())}"
)

print(
    f"Total years: {len(data)}"
)


In [ ]:
# 5. CHECK DATA

if len(data) < 10:
    raise ValueError(
        "Dataset มีจำนวนน้อยเกินไป "
        "ควรมีอย่างน้อย 10 ปี"
    )

latest_year = int(
    data["Year"].max()
)


In [ ]:
# 6. TRAIN / TEST SPLIT

test_size = max(
    3,
    int(len(data) * TEST_RATIO)
)

train = data.iloc[
    :-test_size
].copy()

test = data.iloc[
    -test_size:
].copy()


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    f"Train : "
    f"{int(train['Year'].min())} - "
    f"{int(train['Year'].max())}"
)

print(
    f"Test  : "
    f"{int(test['Year'].min())} - "
    f"{int(test['Year'].max())}"
)


In [ ]:
# 7. FEATURES

X_train = train[
    ["Year"]
]

y_train = train[
    "Temperature"
]

X_test = test[
    ["Year"]
]

y_test = test[
    "Temperature"
]



In [ ]:
# 8. MODEL 1
# LINEAR REGRESSION

print("\n" + "=" * 70)
print("MODEL 1 : LINEAR REGRESSION")
print("=" * 70)

linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_pred = linear_model.predict(
    X_test
)

linear_mae = mean_absolute_error(
    y_test,
    linear_pred
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_pred
    )
)

print(
    f"MAE  : {linear_mae:.4f}"
)

print(
    f"RMSE : {linear_rmse:.4f}"
)


In [ ]:
# 9. MODEL 2
# RANDOM FOREST

print("\n" + "=" * 70)
print("MODEL 2 : RANDOM FOREST")
print("=" * 70)

rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_pred
    )
)

print(
    f"MAE  : {rf_mae:.4f}"
)

print(
    f"RMSE : {rf_rmse:.4f}"
)

In [ ]:
# 10. COMPARE MODELS

comparison = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "Random Forest"
    ],

    "MAE": [
        linear_mae,
        rf_mae
    ],

    "RMSE": [
        linear_rmse,
        rf_rmse
    ]

})

# เรียงจาก RMSE ต่ำไปสูง
comparison = comparison.sort_values(
    "RMSE"
).reset_index(drop=True)


print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format
        }
    )
)



In [ ]:
# 11. SAVE MODEL COMPARISON

comparison_file = (
    "Thailand_Model_Comparison.csv"
)

comparison.to_csv(
    comparison_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved: {comparison_file}"
)


In [ ]:
# 12. SELECT BEST MODEL

best_model = comparison.iloc[0]["Model"]

print("\n" + "=" * 70)
print("MODEL SELECTED")
print("=" * 70)

print(
    f"Selected Model: {best_model}"
)

print(
    "เกณฑ์การเลือก: RMSE ต่ำกว่า"
)



In [ ]:
# 13. TRAIN FINAL MODELS
# ใช้ข้อมูลทั้งหมด

X_all = data[
    ["Year"]
]

y_all = data[
    "Temperature"
]


# ------------------------------------------------------------
# Linear Regression
# ------------------------------------------------------------

final_linear = LinearRegression()

final_linear.fit(
    X_all,
    y_all
)


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

final_rf = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_rf.fit(
    X_all,
    y_all
)



In [ ]:
# 14. CREATE FUTURE YEARS

future_years = np.arange(
    latest_year + 1,
    latest_year + FORECAST_YEARS + 1
)

future_X = pd.DataFrame({
    "Year": future_years
})


print("\nForecast years:")

for year in future_years:
    print(year)


In [ ]:
# 15. FORECAST

linear_forecast = final_linear.predict(
    future_X
)

rf_forecast = final_rf.predict(
    future_X
)


In [ ]:
# 16. SELECT FORECAST

if best_model == "Linear Regression":

    selected_forecast = linear_forecast

else:

    selected_forecast = rf_forecast


In [ ]:
# 17. CREATE FORECAST DATAFRAME

forecast_df = pd.DataFrame({

    "Year": future_years,

    "Linear_Regression": linear_forecast,

    "Random_Forest": rf_forecast,

    "Selected_Model": selected_forecast

})


forecast_df[
    "Selected_Model_Name"
] = best_model


# ปัดทศนิยม
for col in [
    "Linear_Regression",
    "Random_Forest",
    "Selected_Model"
]:

    forecast_df[col] = forecast_df[
        col
    ].round(3)



In [ ]:
# 18. SAVE FORECAST

forecast_file = (
    "Thailand_Temperature_Forecast_2024_2028.csv"
)

forecast_df.to_csv(
    forecast_file,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# 19. DISPLAY FORECAST

print("\n" + "=" * 70)
print("FORECAST RESULT")
print("=" * 70)

print(
    forecast_df.to_string(
        index=False
    )
)

print(
    f"\nSaved: {forecast_file}"
)



In [ ]:
# 20. CREATE GRAPH

plt.figure(
    figsize=(12, 6)
)

# Actual
plt.plot(
    data["Year"],
    data["Temperature"],
    marker="o",
    label="Actual"
)

# Linear Regression
plt.plot(
    future_years,
    linear_forecast,
    marker="o",
    linestyle="--",
    label="Linear Regression"
)

# Random Forest
plt.plot(
    future_years,
    rf_forecast,
    marker="o",
    linestyle="--",
    label="Random Forest"
)

# จุดเริ่ม Forecast
plt.axvline(
    latest_year,
    linestyle=":",
    label="Forecast Start"
)

plt.title(
    "Thailand Temperature Forecast"
)

plt.xlabel(
    "Year"
)

plt.ylabel(
    "Temperature (°C)"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


graph_file = (
    "Thailand_Temperature_Forecast.png"
)

plt.savefig(
    graph_file,
    dpi=300
)

plt.show()



In [ ]:
# 21. FINAL SUMMARY

print("\n" + "=" * 70)
print("COMPLETED")
print("=" * 70)

print(
    f"Latest Dataset Year : {latest_year}"
)

print(
    f"Forecast            : "
    f"{latest_year + 1} - "
    f"{latest_year + FORECAST_YEARS}"
)

print(
    f"Best Model          : {best_model}"
)

print(
    f"\nOutput files:"
)

print(
    f"1. {forecast_file}"
)

print(
    f"2. {comparison_file}"
)

print(
    f"3. {graph_file}"
)

print("\nDone.")